### ingest 8 weeks of most recent data to understand data structure

In [3]:
import requests
import json
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os

load_dotenv()  # loads variables from .env into the environment
api_key = os.getenv("NYT_BOOKS_API_KEY")
output_dir = '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/external/nyt_bestsellers/raw/'

output_file = os.path.join(output_dir, "nyt_books_data_sample.json")

# The Books API uses specific published dates (every Friday/Saturday)
# We'll fetch the last ~8 weekly lists (~2 months)
WEEKS_BACK = 8
LIST_NAME  = "hardcover-fiction"   # change to any list you like
# ────────────────────────────────────────────────────

def get_dates(weeks_back):
    """Generate a list of weekly dates going back N weeks."""
    dates = []
    today = datetime.today()
    for i in range(weeks_back):
        d = today - timedelta(weeks=i)
        dates.append(d.strftime("%Y-%m-%d"))
    return dates

def fetch_list(list_name, date):
    """Fetch one weekly bestseller list from the API."""
    url = f"https://api.nytimes.com/svc/books/v3/lists/{date}/{list_name}.json"
    params = {"api-key": api_key}
    response = requests.get(url, params=params)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"  Error {response.status_code} for date {date}: {response.text[:100]}")
        return None

def main():
    dates = get_dates(WEEKS_BACK)
    all_results = []

    print(f"Fetching '{LIST_NAME}' list for {WEEKS_BACK} weeks...\n")

    for date in dates:
        print(f"  Fetching {date}...")
        data = fetch_list(LIST_NAME, date)
        if data:
            all_results.append(data)
        time.sleep(15)   # rate limited to 5 requests/ min

    # Save to JSON
    os.makedirs(output_dir, exist_ok=True)   # ← add this line
    with open(output_file, "w") as f:
        json.dump(all_results, f, indent=2)

    print(f"\nDone! Saved {len(all_results)} weeks of data to '{output_file}'")

if __name__ == "__main__":
    main()

Fetching 'hardcover-fiction' list for 8 weeks...

  Fetching 2026-04-07...
  Fetching 2026-03-31...
  Fetching 2026-03-24...
  Fetching 2026-03-17...
  Fetching 2026-03-10...
  Fetching 2026-03-03...
  Fetching 2026-02-24...
  Fetching 2026-02-17...

Done! Saved 8 weeks of data to '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/external/nyt_bestsellers/raw/nyt_books_data_sample.json'


In [7]:
#look at the data

import pandas as pd

# Load the JSON file
with open(output_file, "r") as f:
    raw = json.load(f)

# The books are nested inside each week's results
books = []
for week in raw:
    books.extend(week["results"]["books"])

books = pd.DataFrame(books)
books.head()

books[books['title']=='JUDGE STONE']

,age_group,amazon_product_url,article_chapter_link,asterisk,author,book_image,book_image_height,book_image_width,book_review_link,book_uri,...,primary_isbn13,publisher,rank,rank_last_week,sunday_review_link,title,updated_date,weeks_on_list,isbns,buy_links
0,,https://www.amazon.com/dp/0316579831?tag=thene...,,0,Viola Davis and James Patterson,https://static01.nyt.com/bestsellers/images/97...,500,322,,nyt://book/6f72e477-90a5-5cbc-9f2d-d96c1590adeb,...,9780316579834,"Little, Brown and JVL",1,1,,JUDGE STONE,2026-03-18T22:38:16.974Z,2,"[{'isbn10': '', 'isbn13': '9780316579834'}]","[{'name': 'Amazon', 'url': 'https://www.amazon..."
15,,https://www.amazon.com/dp/0316579831?tag=thene...,,0,Viola Davis and James Patterson,https://static01.nyt.com/bestsellers/images/97...,500,322,,nyt://book/6f72e477-90a5-5cbc-9f2d-d96c1590adeb,...,9780316579834,"Little, Brown and JVL",1,0,,JUDGE STONE,2026-03-18T22:38:16.974Z,1,"[{'isbn10': '', 'isbn13': '9780316579834'}]","[{'name': 'Amazon', 'url': 'https://www.amazon..."


In [ ]:
### ingest the full 8 years

import requests
import json
import time
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("NYT_BOOKS_API_KEY")

output_dir = '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/external/nyt_bestsellers/raw/'
output_file = os.path.join(output_dir, "nyt_books_2017_2025.json")

LIST_NAME = "hardcover-fiction"
START_YEAR = 2017
END_YEAR = 2025

def generate_weekly_dates(start_year, end_year):
    """Generate one date per week from Jan 1 of start_year to Dec 31 of end_year."""
    from datetime import datetime, timedelta
    dates = []
    current = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    while current <= end:
        dates.append(current.strftime("%Y-%m-%d"))
        current += timedelta(weeks=1)
    return dates

def fetch_list(list_name, date):
    """Fetch one weekly bestseller list from the API."""
    url = f"https://api.nytimes.com/svc/books/v3/lists/{date}/{list_name}.json"
    params = {"api-key": api_key}
    response = requests.get(url, params=params)

    if response.status_code == 200:
        return response.json()
    elif response.status_code == 429:
        print(f"  Rate limited on {date} — waiting 60s...")
        time.sleep(60)
        return fetch_list(list_name, date)  # retry once after waiting
    else:
        print(f"  Error {response.status_code} for {date}: {response.text[:100]}")
        return None

def main():
    dates = generate_weekly_dates(START_YEAR, END_YEAR)
    all_results = []
    total = len(dates)

    print(f"Fetching '{LIST_NAME}' for {total} weeks ({START_YEAR}–{END_YEAR})...\n")

    for i, date in enumerate(dates, 1):
        print(f"  [{i}/{total}] Fetching {date}...")
        data = fetch_list(LIST_NAME, date)
        if data:
            all_results.append(data)
        time.sleep(12)  # stay under 5 requests/minute

    os.makedirs(output_dir, exist_ok=True)
    with open(output_file, "w") as f:
        json.dump(all_results, f, indent=2)

    print(f"\nDone! Saved {len(all_results)} weeks of data to '{output_file}'")

if __name__ == "__main__":
    main()

In [19]:
## see what lists are available, when they started populating, and the frequency they are updated

url = "https://api.nytimes.com/svc/books/v3/lists/overview.json"
response = requests.get(url, params={"api-key": api_key})

import requests
import json
import time
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta

load_dotenv()
api_key = os.getenv("NYT_BOOKS_API_KEY")

output_dir = '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/external/nyt_bestsellers/raw/'
output_file = os.path.join(output_dir, "nyt_books_overview_2017_2025.json")

START_YEAR = 2017
END_YEAR = 2025

def generate_weekly_dates(start_year, end_year):
    dates = []
    current = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    while current <= end:
        dates.append(current.strftime("%Y-%m-%d"))
        current += timedelta(weeks=1)
    return dates

def fetch_overview(date):
    url = "https://api.nytimes.com/svc/books/v3/lists/overview.json"
    params = {"api-key": api_key, "published_date": date}
    response = requests.get(url, params=params)

    if response.status_code == 200:
        return response.json()
    elif response.status_code == 429:
        print(f"  Rate limited on {date} — waiting 60s...")
        time.sleep(60)
        return fetch_overview(date)
    else:
        print(f"  Error {response.status_code} for {date}: {response.text[:100]}")
        return None

def main():
    dates = generate_weekly_dates(START_YEAR, END_YEAR)
    all_results = []
    total = len(dates)

    print(f"Fetching overview for {total} weeks ({START_YEAR}–{END_YEAR})...\n")

    for i, date in enumerate(dates, 1):
        print(f"  [{i}/{total}] Fetching {date}...")
        data = fetch_overview(date)
        if data:
            all_results.append(data)
        time.sleep(12)

    os.makedirs(output_dir, exist_ok=True)
    with open(output_file, "w") as f:
        json.dump(all_results, f, indent=2)

    print(f"\nDone! Saved {len(all_results)} weeks of data to '{output_file}'")

if __name__ == "__main__":
    main()

Fetching overview for 470 weeks (2017–2025)...

  [1/470] Fetching 2017-01-01...
  [2/470] Fetching 2017-01-08...
  [3/470] Fetching 2017-01-15...
  [4/470] Fetching 2017-01-22...
  [5/470] Fetching 2017-01-29...
  [6/470] Fetching 2017-02-05...
  [7/470] Fetching 2017-02-12...
  [8/470] Fetching 2017-02-19...
  [9/470] Fetching 2017-02-26...
  [10/470] Fetching 2017-03-05...
  [11/470] Fetching 2017-03-12...
  [12/470] Fetching 2017-03-19...
  [13/470] Fetching 2017-03-26...
  [14/470] Fetching 2017-04-02...
  [15/470] Fetching 2017-04-09...
  [16/470] Fetching 2017-04-16...
  [17/470] Fetching 2017-04-23...
  [18/470] Fetching 2017-04-30...
  [19/470] Fetching 2017-05-07...
  [20/470] Fetching 2017-05-14...
  [21/470] Fetching 2017-05-21...
  [22/470] Fetching 2017-05-28...
  [23/470] Fetching 2017-06-04...
  [24/470] Fetching 2017-06-11...
  [25/470] Fetching 2017-06-18...
  [26/470] Fetching 2017-06-25...
  [27/470] Fetching 2017-07-02...
  [28/470] Fetching 2017-07-09...
  [29/470

In [26]:
import json
import pandas as pd

# Load the JSON file
with open(output_file, "r") as f:
    raw = json.load(f)

# Flatten: one row per book per list per week
rows = []
for week in raw:
    published_date = week["results"]["published_date"]
    bestsellers_date = week["results"]["bestsellers_date"]
    
    for lst in week["results"]["lists"]:
        list_name = lst["list_name_encoded"]
        display_name = lst["display_name"]
        updated = lst["updated"]
        
        for book in lst["books"]:
            rows.append({
                "published_date":    published_date,
                "bestsellers_date":  bestsellers_date,
                "list_name":         list_name,
                "display_name":      display_name,
                "updated":           updated,
                "rank":              book["rank"],
                "rank_last_week":    book["rank_last_week"],
                "weeks_on_list":     book["weeks_on_list"],
                "title":             book["title"],
                "author":            book["author"],
                "publisher":         book["publisher"],
                "description":       book["description"],
                "primary_isbn13":    book["primary_isbn13"],
            })

df = pd.DataFrame(rows)
df["published_date"] = pd.to_datetime(df["published_date"])
df["bestsellers_date"] = pd.to_datetime(df["bestsellers_date"])

print(df.shape)
print(df.head())

print('\n\n=== Lists Metadata ===')
list_summary = (df.groupby(['list_name', 'display_name', 'updated'])
                .agg(
                    earliest_date=('bestsellers_date', 'min'),
                    latest_date=('bestsellers_date', 'max'),
                    total_weeks=('published_date', 'nunique')
                )
                .reset_index()
                .sort_values('earliest_date'))

print(list_summary.sort_values('total_weeks', ascending = False).to_string(index=False))


(102731, 13)
  published_date bestsellers_date                          list_name  \
0     2017-01-01       2016-12-17  combined-print-and-e-book-fiction   
1     2017-01-01       2016-12-17  combined-print-and-e-book-fiction   
2     2017-01-01       2016-12-17  combined-print-and-e-book-fiction   
3     2017-01-01       2016-12-17  combined-print-and-e-book-fiction   
4     2017-01-01       2016-12-17  combined-print-and-e-book-fiction   

                      display_name updated  rank  rank_last_week  \
0  Combined Print & E-Book Fiction  WEEKLY     1               2   
1  Combined Print & E-Book Fiction  WEEKLY     2               4   
2  Combined Print & E-Book Fiction  WEEKLY     3               5   
3  Combined Print & E-Book Fiction  WEEKLY     4              15   
4  Combined Print & E-Book Fiction  WEEKLY     5               0   

   weeks_on_list               title            author             publisher  \
0              8        THE WHISTLER      John Grisham           

In [29]:
### check one title to see how it shows up

df[df['title'] == 'THE WHISTLER'].sort_values(['list_name', 'bestsellers_date'], ascending = False)

,published_date,bestsellers_date,list_name,display_name,updated,rank,rank_last_week,weeks_on_list,title,author,publisher,description,primary_isbn13
7219,2017-08-20,2017-08-05,trade-fiction-paperback,Paperback Trade Fiction,WEEKLY,10,9,4,THE WHISTLER,John Grisham,Bantam,A whistleblower alerts a Florida investigator ...,9781101967676
7058,2017-08-13,2017-07-29,trade-fiction-paperback,Paperback Trade Fiction,WEEKLY,9,6,3,THE WHISTLER,John Grisham,Bantam,A whistleblower alerts a Florida investigator ...,9781101967676
6895,2017-08-06,2017-07-22,trade-fiction-paperback,Paperback Trade Fiction,WEEKLY,6,9,2,THE WHISTLER,John Grisham,Bantam,A whistleblower alerts a Florida investigator ...,9781101967676
6738,2017-07-30,2017-07-15,trade-fiction-paperback,Paperback Trade Fiction,WEEKLY,9,0,1,THE WHISTLER,John Grisham,Bantam,A whistleblower alerts a Florida investigator ...,9781101967676
4153,2017-04-09,2017-03-25,hardcover-fiction,Hardcover Fiction,WEEKLY,14,11,22,THE WHISTLER,John Grisham,Doubleday,A whistleblower alerts a Florida investigator ...,9780385541190
3990,2017-04-02,2017-03-18,hardcover-fiction,Hardcover Fiction,WEEKLY,11,8,21,THE WHISTLER,John Grisham,Doubleday,A whistleblower alerts a Florida investigator ...,9780385541190
3827,2017-03-26,2017-03-11,hardcover-fiction,Hardcover Fiction,WEEKLY,8,10,20,THE WHISTLER,John Grisham,Doubleday,A whistleblower alerts a Florida investigator ...,9780385541190
3669,2017-03-19,2017-03-04,hardcover-fiction,Hardcover Fiction,WEEKLY,10,12,19,THE WHISTLER,John Grisham,Doubleday,A whistleblower alerts a Florida investigator ...,9780385541190
3511,2017-03-12,2017-02-25,hardcover-fiction,Hardcover Fiction,WEEKLY,12,7,18,THE WHISTLER,John Grisham,Doubleday,A whistleblower alerts a Florida investigator ...,9780385541190
3346,2017-03-05,2017-02-18,hardcover-fiction,Hardcover Fiction,WEEKLY,7,4,17,THE WHISTLER,John Grisham,Doubleday,A whistleblower alerts a Florida investigator ...,9780385541190


In [31]:
import re

def clean_name(text):
    """Normalize to uppercase, strip whitespace, remove years and metadata."""
    if not isinstance(text, str):
        return ""
    text = text.upper().strip()
    text = re.sub(r'\b(18|19|20)\d{2}\b', '', text)  # remove years
    text = re.sub(r'\s+(JR|SR|II|III|IV|MD|PHD|ESQ)\.?\s*$', '', text)  # remove suffixes
    text = re.sub(r'\s+', ' ', text).strip()  # collapse whitespace
    return text

def max_consecutive(week_list):
    """Return longest consecutive streak of week numbers."""
    if not week_list:
        return 0
    weeks = sorted(set(week_list))
    max_streak = streak = 1
    for i in range(1, len(weeks)):
        if weeks[i] == weeks[i-1] + 1:
            streak += 1
            max_streak = max(max_streak, streak)
        else:
            streak = 1
    return max_streak

# Add time columns
df['year']         = df['published_date'].dt.year
df['month']        = df['published_date'].dt.month
df['week_of_year'] = df['published_date'].dt.isocalendar().week.astype(int)

# Build work_key
df['title_clean']  = df['title'].apply(clean_name)
df['author_clean'] = df['author'].apply(clean_name)
df['work_key']     = df['title_clean'] + ' || ' + df['author_clean']

# Monthly aggregate
monthly_nyt = (df.groupby(['year', 'month', 'work_key', 'list_name', 'display_name'])
               .agg(
                   title=('title_clean', 'first'),
                   author=('author_clean', 'first'),
                   weeks_on_list_this_month=('rank', 'count'),
                   peak_rank=('rank', 'min'),
                   avg_rank=('rank', 'mean'),
                   new_entry=('rank_last_week', lambda x: (x == 0).any()),
                   total_weeks_on_list=('weeks_on_list', 'max'),
                   week_numbers=('week_of_year', list)
               )
               .reset_index())

# Add consecutive weeks (requires the list of week numbers)
monthly_nyt['max_consecutive_weeks'] = monthly_nyt['week_numbers'].apply(max_consecutive)
monthly_nyt = monthly_nyt.drop(columns='week_numbers')

# Round avg_rank
monthly_nyt['avg_rank'] = monthly_nyt['avg_rank'].round(2)

# Final column order
monthly_nyt = monthly_nyt[[
    'year', 'month', 'work_key', 'title', 'author',
    'list_name', 'display_name',
    'weeks_on_list_this_month', 'peak_rank', 'avg_rank',
    'max_consecutive_weeks', 'new_entry', 'total_weeks_on_list'
]]

print(monthly_nyt.shape)
print(monthly_nyt.head(20))

(38409, 13)
    year  month                                          work_key  \
0   2017      1      10-DAY GREEN SMOOTHIE CLEANSE || J. J. SMITH   
1   2017      1  15TH AFFAIR || JAMES PATTERSON AND MAXINE PAETRO   
2   2017      1                           5TH WAVE || RICK YANCEY   
3   2017      1     99 || WAYNE GRETZKY WITH KIRSTIE MCLELLAN DAY   
4   2017      1         A COURT OF MIST AND FURY || SARAH J. MAAS   
5   2017      1               A DOG'S PURPOSE || W. BRUCE CAMERON   
6   2017      1               A DOG'S PURPOSE || W. BRUCE CAMERON   
7   2017      1               A DOG'S PURPOSE || W. BRUCE CAMERON   
8   2017      1               A DOG'S PURPOSE || W. BRUCE CAMERON   
9   2017      1    A FIELD GUIDE TO REDHEADS || ELIZABETH GRAEBER   
10  2017      1              A GENTLEMAN IN MOSCOW || AMOR TOWLES   
11  2017      1               A LIFE WELL PLAYED || ARNOLD PALMER   
12  2017      1               A LIFE WELL PLAYED || ARNOLD PALMER   
13  2017      1       

In [32]:
## store the data

processed_dir = '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/external/nyt_bestsellers/processed/'
monthly_nyt.reset_index(drop=True).to_parquet(f"{processed_dir}/monthly_nyt.parquet")
